In [1]:
# ============================================================================
# <IPA> via MAX SECOND DERIVATIVE PER RUN — standalone, single cell
#
# Same elbow rule as ../CE_Avg_method/ipa_2nd_derivative_standalone.ipynb, but
# applied to each raw run separately, then the IPA values are averaged:
#   CE_Avg_method : average CE over runs -> one elbow -> one IPA
#   IPA_Avg_method: one elbow per run   -> one IPA per run -> <IPA>, STD
#
# Pipeline (raw data only, no curve fitting):
#   1. Auto-detect pruning levels from p-percentage_* directories in BASE_DIR
#   2. For each (P%, BS): load every raw run slp_{p}_{bs}_run_*.txt
#      (CE_TEST vs Batch_Number, each run up to its own early stop)
#   3. No step-artifact truncation by default: the step comes from runs
#      dropping out of the average, which a single run does not have.
#      STEP_CUT = True applies the CE_Avg_method rule to each run anyway.
#   4. Elbow per run = data point with max discrete second derivative
#      (identical second_derivative_elbow() to CE_Avg_method)
#   5. IPA_run = |CE_o - CE_learned| / BN_learned,  CE_o = ln(10)
#      <IPA> = mean over valid runs, STD = sample std (ddof=1)
#      Degenerate flat runs (e.g. P=100%) and empty files -> excluded
#   6. Write per-run CSV + summary CSV + <IPA>-vs-P% plot
#
# Requires: numpy, pandas, matplotlib
# ============================================================================
import os, glob, re
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ── Config ────────────────────────────────────────────────────────────────────
BN_STEP_MIN = 100     # step detection starts here (only used if STEP_CUT)
STEP_THRESH = 0.01    # CE jump that flags the averaging artifact
STEP_CUT    = False   # apply the averaged-curve step truncation to each run
SMOOTH_W    = 30       # 1 = raw data; odd int > 1 = moving-average smoothing
CE_o        = np.log(10)
BATCH_SIZES = [64, 1024, 60000]
BS_COLOR    = {64: "#1f77b4", 1024: "#d62728", 60000: "#2ca02c"}

# Paths are relative to this notebook: SLP-MNIST/IPA_2nd_derivative_method/IPA_Avg_method/
BASE_DIR = os.path.abspath(os.path.join("..", "..", "prune_layers_ALL"))
OUT_DIR  = os.path.abspath(".")
# ──────────────────────────────────────────────────────────────────────────────

PRUNING_LEVELS = sorted(float(re.search(r"p-percentage_([\d.]+)", d).group(1))
                        for d in glob.glob(os.path.join(BASE_DIR, "p-percentage_*")))
print(f"Found {len(PRUNING_LEVELS)} pruning levels  |  SMOOTH_W={SMOOTH_W} "
      f"({'raw data' if SMOOTH_W <= 1 else 'moving average'})  |  STEP_CUT={STEP_CUT}")


def step_cut(bns, ces):
    """Cut at the first BN >= BN_STEP_MIN where |CE[i] - CE[i-1]| > STEP_THRESH."""
    cutoff_BN = float(bns[-1])
    for i in range(1, len(bns)):
        if bns[i] >= BN_STEP_MIN and abs(ces[i] - ces[i - 1]) > STEP_THRESH:
            cutoff_BN = float(bns[i])
            break
    m = bns < cutoff_BN
    return bns[m], ces[m]


def load_run(path):
    """Raw CE_TEST curve of one run (whitespace-separated .txt); None if empty/bad."""
    try:
        df = pd.read_csv(path, sep=r"\s+")
    except pd.errors.EmptyDataError:
        return None
    if "CE_TEST" not in df.columns or "Batch_Number" not in df.columns:
        return None
    df = df[["Batch_Number", "CE_TEST"]].apply(pd.to_numeric, errors="coerce").dropna()
    bns = df["Batch_Number"].values.astype(float)
    ces = df["CE_TEST"].values.astype(float)
    return step_cut(bns, ces) if STEP_CUT else (bns, ces)


def run_files(p, bs):
    """Raw run files for (p, bs), sorted by run number."""
    files = glob.glob(os.path.join(BASE_DIR, f"p-percentage_{p}", f"batch_size_{bs}",
                                   f"slp_{p}_{bs}_run_*.txt"))
    return sorted(files, key=lambda f: int(re.search(r"_run_(\d+)\.txt$", f).group(1)))


def second_derivative_elbow(BN, CE, smooth_w=SMOOTH_W):
    """Max-second-derivative elbow on the data. -> (BN_learned, CE_learned, IPA)"""
    BN = np.asarray(BN, float)
    CE = np.asarray(CE, float)
    if len(BN) < max(5, smooth_w + 2) or np.ptp(CE) <= 1e-10:
        return np.nan, np.nan, np.nan                 # degenerate (e.g. P=100%)
    if smooth_w > 1:
        y = np.convolve(CE, np.ones(int(smooth_w)) / smooth_w, mode="same")
        h = int(smooth_w) // 2                        # unreliable convolution edges
    else:
        y, h = CE, 0
    d2 = np.gradient(np.gradient(y, BN), BN)          # discrete 2nd derivative
    lo, hi = 1 + h, len(BN) - 1 - h                   # exclude endpoints (+ edges)
    if hi <= lo:
        return np.nan, np.nan, np.nan
    i = lo + int(np.argmax(d2[lo:hi]))
    if BN[i] <= 0:
        return np.nan, np.nan, np.nan
    return float(BN[i]), float(CE[i]), abs(CE_o - CE[i]) / BN[i]


# ── Per-run elbows for all (P%, BS) ───────────────────────────────────────────
run_rows = []
for p in PRUNING_LEVELS:
    for bs in BATCH_SIZES:
        for f in run_files(p, bs):
            run = int(re.search(r"_run_(\d+)\.txt$", f).group(1))
            curve = load_run(f)
            bn_l, ce_l, ipa = second_derivative_elbow(*curve) if curve is not None else (np.nan,)*3
            run_rows.append({"P%": p * 100, "BS": bs, "Run": run,
                             "BN_learned": bn_l, "CE_learned": ce_l, "IPA": ipa})
per_run_df = pd.DataFrame(run_rows)

per_run_path = os.path.join(OUT_DIR, "ipa_per_run_2nd_deriv.csv")
per_run_df.to_csv(per_run_path, index=False)
print(f"Saved: {per_run_path}  ({len(per_run_df)} runs)")

# ── Average IPA over runs, build summary ──────────────────────────────────────
rows = []
for p in PRUNING_LEVELS:
    row = {"P%": p * 100}
    for bs in BATCH_SIZES:
        sub = per_run_df[(per_run_df["P%"] == p * 100) & (per_run_df["BS"] == bs)]
        ok  = sub.dropna(subset=["IPA"])
        row[f"n_runs_{bs}"]          = len(sub)
        row[f"n_valid_{bs}"]         = len(ok)
        row[f"BN_learned_mean_{bs}"] = ok["BN_learned"].mean() if len(ok) else np.nan
        row[f"CE_learned_mean_{bs}"] = ok["CE_learned"].mean() if len(ok) else np.nan
        row[f"IPA_Avg_{bs}"]         = ok["IPA"].mean() if len(ok) else np.nan
        row[f"STD_{bs}"]             = ok["IPA"].std(ddof=1) if len(ok) >= 2 else np.nan
    rows.append(row)
summary_df = pd.DataFrame(rows)

csv_path = os.path.join(OUT_DIR, "ipa_summary_2nd_deriv_ipa_avg.csv")
summary_df.to_csv(csv_path, index=False)
print(f"Saved: {csv_path}")
print(summary_df[["P%"] + [c for b in BATCH_SIZES
                           for c in (f"n_valid_{b}", f"BN_learned_mean_{b}", f"IPA_Avg_{b}", f"STD_{b}")]]
      .to_string(index=False))

# ── Plot <IPA> vs P% (error bars = STD over runs) ─────────────────────────────
plt.rcParams.update({"font.size": 13})
fig, ax = plt.subplots(figsize=(9, 5.5))
for bs in BATCH_SIZES:
    sub = summary_df.dropna(subset=[f"IPA_Avg_{bs}"])
    ax.errorbar(sub["P%"], sub[f"IPA_Avg_{bs}"], yerr=sub[f"STD_{bs}"], fmt="^-.",
                color=BS_COLOR[bs], ms=6, lw=2, capsize=3, label=f"BS={bs}")
ax.set_xlabel("Pruning Percentage (%)")
ax.set_ylabel("<IPA> over runs")
smooth_tag = "raw data" if SMOOTH_W <= 1 else f"moving avg w={SMOOTH_W}"
ax.set_title(f"<IPA> vs Pruning — per-run max 2nd derivative elbow ({smooth_tag})")
ax.grid(True, alpha=0.3)
ax.legend(frameon=False)
png_path = os.path.join(OUT_DIR, "ipa_plot_2nd_deriv_ipa_avg.png")
plt.tight_layout()
plt.savefig(png_path, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {png_path}")

Found 19 pruning levels  |  SMOOTH_W=30 (moving average)  |  STEP_CUT=False


Saved: C:\Users\lpand003\Desktop\Research\ipa-pruning\physlab\SLP\SLP-MNIST\IPA_2nd_derivative_method\IPA_Avg_method\ipa_per_run_2nd_deriv.csv  (4803 runs)
Saved: C:\Users\lpand003\Desktop\Research\ipa-pruning\physlab\SLP\SLP-MNIST\IPA_2nd_derivative_method\IPA_Avg_method\ipa_summary_2nd_deriv_ipa_avg.csv
   P%  n_valid_64  BN_learned_mean_64  IPA_Avg_64   STD_64  n_valid_1024  BN_learned_mean_1024  IPA_Avg_1024  STD_1024  n_valid_60000  BN_learned_mean_60000  IPA_Avg_60000  STD_60000
  0.0         100           17.160000    0.091623 0.002204           100             17.000000      0.099318  0.001219            100              17.000000       0.098507   0.000979
 10.0         100           17.160000    0.090128 0.002416           100             17.000000      0.098041  0.001102            100              17.000000       0.097699   0.000973
 20.0         100           17.160000    0.087999 0.002386           100             17.000000      0.096691  0.001037            100           

Saved: C:\Users\lpand003\Desktop\Research\ipa-pruning\physlab\SLP\SLP-MNIST\IPA_2nd_derivative_method\IPA_Avg_method\ipa_plot_2nd_deriv_ipa_avg.png
